In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

# Policy network
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.fc(state)

# Environment and setup
env = gym.make("CartPole-v1")
policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
optimizer = optim.Adam(policy.parameters(), lr=1e-2)
gamma = 0.99

for episode in range(500):
    states, actions, rewards = [], [], []
    state = env.reset()[0]
    done = False

    # 1️⃣ Run an episode
    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32)
        probs = policy(state_tensor)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()

        next_state, reward, done, _, _ = env.step(action.item())
        states.append(state_tensor)
        actions.append(action)
        rewards.append(reward)
        state = next_state

    # 2️⃣ Compute returns (discounted rewards)
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # normalize

    # 3️⃣ Compute policy loss
    log_probs = []
    for state, action in zip(states, actions):
        probs = policy(state)
        dist = torch.distributions.Categorical(probs)
        log_probs.append(dist.log_prob(action))
    log_probs = torch.stack(log_probs)

    loss = -(log_probs * returns).sum()

    # 4️⃣ Backpropagate
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Episode {episode}, total reward: {sum(rewards):.2f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

# --- Policy Network ---
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.net(state)

# --- REINFORCE Agent ---
def reinforce(env_name="CartPole-v1", lr=1e-3, gamma=0.99, episodes=500):
    env = gym.make(env_name)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    for ep in range(episodes):
        log_probs, rewards = [], []
        state = env.reset()[0] if isinstance(env.reset(), tuple) else env.reset()

        done = False
        while not done:
            state_t = torch.tensor(state, dtype=torch.float32)
            probs = policy(state_t)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))

            next_state, reward, done, *_ = env.step(action.item())
            rewards.append(reward)
            state = next_state

        # Compute discounted returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # Compute policy gradient loss
        loss = -torch.sum(torch.stack(log_probs) * returns)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Episode {ep+1}: total reward = {sum(rewards)}")

    env.close()

reinforce()


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

def scrape_interesting_engineering(max_pages=100):
    # Chrome options
    options = Options()
    options.add_argument("--headless")  # comment this line if you want to see the browser
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    # Start driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get("https://interestingengineering.com/news")

    articles = []
    page = 0

    while page < max_pages:
        page += 1
        print(f"📰 Scraping page {page}...")

        # Wait for articles to load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "article"))
            )
        except Exception:
            print("❌ No articles found, stopping.")
            break

        # Parse HTML
        soup = BeautifulSoup(driver.page_source, "html.parser")
        found = 0
        for article in soup.select("article"):
            title_tag = article.find("a")
            date_tag = article.find("time")
            if title_tag:
                title = title_tag.get_text(strip=True)
                date = date_tag.get_text(strip=True) if date_tag else "No date"
                articles.append((title, date))
                found += 1
        print(f"  Found {found} articles on this page.")

        # Try to click NEXT
        try:
            next_btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, "//a[contains(translate(., 'next', 'NEXT'), 'NEXT')]"))
            )
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(2)  # wait for next page to load
        except Exception:
            print("✅ Reached last page.")
            break

    driver.quit()
    return articles


def save_to_text(articles, filename="InterestingEngineering_News.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        for title, date in articles:
            f.write(f"{title} — {date}\n")
    print(f"✅ Saved {len(articles)} articles to {filename}")


if __name__ == "__main__":
    data = scrape_interesting_engineering()
    save_to_text(data)


📰 Scraping page 1...
  Found 1 articles on this page.
📰 Scraping page 2...
  Found 1 articles on this page.
📰 Scraping page 3...
  Found 10 articles on this page.
✅ Reached last page.


RemoteDisconnected: Remote end closed connection without response